# CSA08 Assignment 4 – Robust Data Processing with Files, Exceptions, Modules and Packages

**Student Name:** Darshan A  
**Register Number:** 1973260011  
**Course Code:** CSA0812  
**Task 2:** From Requirement to File-Processing Design: Biometric Attendance Files  
**Marks:** 8 | **Bloom’s Level:** K3 – Apply

---

## Aim
To design a Python file-processing solution that reads daily biometric attendance files, calculates each student's monthly attendance percentage, identifies students below 75%, and records damaged lines with their source filename and line number.

## Objectives
- Describe the input and output file formats.
- Select suitable file modes, encodings and file-reading/writing methods.
- Design an algorithm to process all daily files in a month.
- Trace the algorithm using a small file containing a damaged record.


## Task 2.1 – Model the Data (2 Marks)

### Input file format
Each daily text file contains one attendance record per line, with three comma-separated fields:

`RegisterNumber,Date,Status`

- **RegisterNumber:** student's register number.
- **Date:** attendance date in `YYYY-MM-DD` format.
- **Status:** `P` for Present or `A` for Absent.

Example:

```text
192311001,2026-10-01,P
192311002,2026-10-01,A
192311003,2026-10-01,P
```

A damaged record may have missing fields or an invalid status, for example:

```text
192311004,2026-10-01
192311005,2026-10-01,L
```

### Data structures
A dictionary named `attendance` is used to accumulate counts for each register number. Each dictionary value stores the number of present and absent days.

Example:

```python
attendance = {
    "192311001": {"present": 2, "absent": 1},
    "192311002": {"present": 1, "absent": 2}
}
```

A list named `damaged_records` stores the filename, line number and original damaged line.

### Output files
1. **monthly_attendance_report.csv** – contains each register number, present days, absent days, attendance percentage and a remark showing whether the student is below 75%.
2. **damaged_lines.log** – contains the source filename, line number and content of each damaged record.

The attendance percentage is calculated as:

\[
\text{Attendance Percentage} =
\frac{\text{Present Days}}{\text{Present Days} + \text{Absent Days}} \times 100
\]


## Task 2.2 – Design the File Operations (2 Marks)

| File operation | Mode / method | Reason |
|---|---|---|
| Read each daily attendance file | `r` | Opens existing daily files without changing them. |
| Read records | File iteration (`for line in file`) | Processes one record at a time and avoids loading the whole file. |
| Write the monthly report | `w` | Creates a fresh report for the current processing run. |
| Write damaged-record log | `w` | Creates a fresh log so that it matches the current run. |
| Write output records | `write()` | Writes formatted report and log lines. |
| Encoding | `utf-8` | Provides consistent text decoding and encoding. |

The program uses `with open(...)` so that each file is closed automatically after use. The `read()` method is not necessary because the files can be processed by iteration, one line at a time.


## Task 2.3 – Design the Processing (3 Marks)

### Pseudocode

**Algorithm: Monthly Biometric Attendance Processing**

**Input:** Folder containing daily attendance files for one month.

**Output:** Monthly attendance report and damaged-record log.

1. START.
2. Set the input folder path.
3. Create an empty dictionary `attendance`.
4. Create an empty list `damaged_records`.
5. Obtain the list of daily attendance files from the folder using `pathlib`.
6. FOR each file in the list:
   1. Open the file in read mode using UTF-8 encoding.
   2. Set the line number to zero.
   3. FOR each line in the file:
      1. Increase the line number by one.
      2. Remove surrounding whitespace and split the line using comma.
      3. IF the number of fields is not three:
         - Add filename, line number and original line to `damaged_records`.
         - Continue with the next line.
      4. Extract register number, date and status.
      5. IF register number or date is empty, or status is neither `P` nor `A`:
         - Add filename, line number and original line to `damaged_records`.
         - Continue with the next line.
      6. IF register number is not present in `attendance`, initialise its present and absent counts to zero.
      7. IF status is `P`, increase the present count by one; otherwise increase the absent count by one.
   3. Close the input file.
7. Open the monthly report in write mode.
8. Write the report heading.
9. FOR each register number in `attendance`:
   1. Calculate total valid days.
   2. Calculate attendance percentage.
   3. IF percentage is below 75, mark the student as `Below 75%`; otherwise mark as `OK`.
   4. Write the student's counts, percentage and remark.
10. Open the damaged-record log in write mode.
11. Write every damaged record with its filename and line number.
12. Close both output files.
13. STOP.

### Processing decisions
- Every daily file is processed once.
- Only valid `P` and `A` records contribute to attendance counts.
- Damaged records are excluded from the percentage calculation and are written to the error log.
- The monthly report is recreated each time the program runs.


## Create Sample Daily Attendance Files

The next cell creates a small set of daily files for October 2026. The sample includes valid attendance records and damaged records so that the processing and error log can be demonstrated.


In [1]:
from pathlib import Path

# Create a folder for the daily attendance files
input_folder = Path("attendance_daily")
input_folder.mkdir(exist_ok=True)

daily_data = {
    "attendance_2026-10-01.txt": (
        "192311001,2026-10-01,P\n"
        "192311002,2026-10-01,A\n"
        "192311003,2026-10-01,P\n"
        "192311004,2026-10-01\n"
    ),
    "attendance_2026-10-02.txt": (
        "192311001,2026-10-02,P\n"
        "192311002,2026-10-02,P\n"
        "192311003,2026-10-02,A\n"
        "192311004,2026-10-02,L\n"
    ),
    "attendance_2026-10-03.txt": (
        "192311001,2026-10-03,A\n"
        "192311002,2026-10-03,A\n"
        "192311003,2026-10-03,P\n"
    )
}

for filename, content in daily_data.items():
    with open(input_folder / filename, "w", encoding="utf-8") as file:
        file.write(content)

print("Sample daily attendance files created successfully.")
print("Files:", sorted(path.name for path in input_folder.glob("*.txt")))


Sample daily attendance files created successfully.
Files: ['attendance_2026-10-01.txt', 'attendance_2026-10-02.txt', 'attendance_2026-10-03.txt']


## Python Implementation – Monthly Attendance Processor

The program below uses `pathlib` to list the daily files, a dictionary to accumulate attendance counts, and two output files for the monthly report and damaged-record log.


In [2]:
from pathlib import Path
import csv


def process_monthly_attendance(input_folder, report_file, error_log):
    input_folder = Path(input_folder)
    attendance = {}
    damaged_records = []

    # Read every daily attendance file exactly once
    daily_files = sorted(input_folder.glob("*.txt"))

    for daily_file in daily_files:
        with open(daily_file, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                original_line = line.rstrip("\n")
                parts = line.strip().split(",")

                # Check the number of fields
                if len(parts) != 3:
                    damaged_records.append(
                        (daily_file.name, line_number, original_line,
                         "Expected exactly 3 fields")
                    )
                    continue

                register_number, date, status = [part.strip() for part in parts]

                # Validate required fields and attendance status
                if not register_number or not date or status not in ("P", "A"):
                    damaged_records.append(
                        (daily_file.name, line_number, original_line,
                         "Missing field or invalid status")
                    )
                    continue

                # Initialise the student's counts when first encountered
                if register_number not in attendance:
                    attendance[register_number] = {
                        "present": 0,
                        "absent": 0
                    }

                # Accumulate attendance
                if status == "P":
                    attendance[register_number]["present"] += 1
                else:
                    attendance[register_number]["absent"] += 1

    # Write the monthly attendance report
    with open(report_file, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([
            "Register Number", "Present Days", "Absent Days",
            "Attendance Percentage", "Remark"
        ])

        for register_number in sorted(attendance):
            present = attendance[register_number]["present"]
            absent = attendance[register_number]["absent"]
            total_days = present + absent
            percentage = (present / total_days) * 100 if total_days else 0

            remark = "Below 75%" if percentage < 75 else "OK"

            writer.writerow([
                register_number, present, absent,
                f"{percentage:.2f}", remark
            ])

    # Write the damaged-record log
    with open(error_log, "w", encoding="utf-8") as file:
        file.write("DAMAGED ATTENDANCE RECORDS\n")
        file.write("==========================\n")

        for filename, line_number, content, reason in damaged_records:
            file.write(
                f"File: {filename} | Line: {line_number} | "
                f"Record: {content} | Reason: {reason}\n"
            )

    print("Daily files processed:", len(daily_files))
    print("Students processed:", len(attendance))
    print("Damaged records:", len(damaged_records))
    print("Monthly report saved to:", report_file)
    print("Damaged-record log saved to:", error_log)

    return attendance, damaged_records


# Run the monthly attendance processor
attendance, damaged_records = process_monthly_attendance(
    "attendance_daily",
    "monthly_attendance_report.csv",
    "damaged_lines.log"
)


Daily files processed: 3
Students processed: 3
Damaged records: 2
Monthly report saved to: monthly_attendance_report.csv
Damaged-record log saved to: damaged_lines.log


## Display the Generated Output Files

The next cell displays both output files so that the report and damaged-record log can be checked in the notebook.


In [3]:
print("MONTHLY ATTENDANCE REPORT")
print("=========================")

with open("monthly_attendance_report.csv", "r", encoding="utf-8") as file:
    print(file.read())

print("DAMAGED RECORD LOG")
print("==================")

with open("damaged_lines.log", "r", encoding="utf-8") as file:
    print(file.read())


MONTHLY ATTENDANCE REPORT
Register Number,Present Days,Absent Days,Attendance Percentage,Remark
192311001,2,1,66.67,Below 75%
192311002,1,2,33.33,Below 75%
192311003,2,1,66.67,Below 75%

DAMAGED RECORD LOG
DAMAGED ATTENDANCE RECORDS
File: attendance_2026-10-01.txt | Line: 4 | Record: 192311004,2026-10-01 | Reason: Expected exactly 3 fields
File: attendance_2026-10-02.txt | Line: 4 | Record: 192311004,2026-10-02,L | Reason: Missing field or invalid status



## Task 2.4 – Trace (1 Mark)

Consider the following small input file named `attendance_day1.txt`:

| Line number | Record | Processing result |
|---:|---|---|
| 1 | `192311001,2026-10-01,P` | Valid. Present count for 192311001 becomes 1. |
| 2 | `192311002,2026-10-01,A` | Valid. Absent count for 192311002 becomes 1. |
| 3 | `192311003,2026-10-01` | Damaged. Only two fields are present; record is logged with filename and line number. |
| 4 | `192311004,2026-10-01,L` | Damaged. Status `L` is not `P` or `A`; record is logged with filename and line number. |

### Trace result
- Register number `192311001`: Present = 1, Absent = 0, Attendance = 100%.
- Register number `192311002`: Present = 0, Absent = 1, Attendance = 0%.
- Two damaged records are written to the error log.
- The damaged records do not affect the attendance percentages.


## Verify the Results

The next cell checks the accumulated counts and confirms that the damaged records include the correct filenames and line numbers.


In [4]:
# Automatic verification checks

assert attendance["192311001"] == {"present": 2, "absent": 1}
assert attendance["192311002"] == {"present": 1, "absent": 2}
assert attendance["192311003"] == {"present": 2, "absent": 1}
assert attendance["192311004"] == {"present": 0, "absent": 0} if "192311004" in attendance else True

# There are two damaged records in the sample files
assert len(damaged_records) == 2
assert damaged_records[0][0] == "attendance_2026-10-01.txt"
assert damaged_records[0][1] == 4
assert damaged_records[1][0] == "attendance_2026-10-02.txt"
assert damaged_records[1][1] == 4

print("All verification checks passed.")
print("Valid students processed:", len(attendance))
print("Damaged records detected:", len(damaged_records))


All verification checks passed.
Valid students processed: 3
Damaged records detected: 2


## Alternative Concept and Complexity Analysis

An alternative method is to calculate attendance separately for each student by re-reading every daily file and checking all lines for that student's register number.

Let:
- `s` = number of students.
- `f` = number of daily files.
- `l` = average number of lines in each file.

### Re-reading files for every student
For each of the `s` students, the program scans all `f` files and examines approximately `l` lines per file. Therefore, the time complexity is:

**O(s × f × l)**

This approach can repeatedly read the same data and takes more processing time as the number of students increases.

### Reading each file once and using a dictionary
The dictionary-based approach scans each line once and updates the relevant student's counts. Its time complexity is:

**O(f × l)** on average, assuming average O(1) dictionary access.

| Factor | Re-reading for each student | One scan with dictionary |
|---|---|---|
| Time complexity | O(s × f × l) | O(f × l) average |
| File scanning | Repeated for every student | Each daily file scanned once |
| Additional data space | O(s) for results | O(s) for the dictionary and O(d) for damaged records |
| Efficiency | Less efficient for many students | More efficient for monthly processing |

Here, `d` represents the number of damaged records. The dictionary method uses memory to retain each student's counts, but avoids repeatedly scanning all the files.

**Conclusion:** Reading each daily file once and accumulating counts in a dictionary is more efficient and suitable for processing monthly attendance for many students.


## Task 2 – Conclusion

The attendance processor reads all daily files from a folder, validates each record and accumulates present and absent counts using a dictionary. It generates a monthly attendance report, identifies students whose attendance is below 75%, and records damaged records with their source filename and line number. The dictionary-based single-pass design avoids repeatedly reading the same files and provides an efficient solution for monthly attendance processing.
